In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from iss_preprocess.io import (
    get_processed_path,
)

from brisc.manuscript_analysis.double_labeling_estimation import (
    run_double_labeling_analysis,
    plot_observed_vs_expected,
    plot_observed_vs_expected_log,
    plot_residuals,
    plot_injection_site,
    sweep_density_thresholds,
    plot_threshold_sweep,
    load_spots_per_barcode,
    spot_count_summary,
    plot_spot_count_distributions,
    run_filtered_analysis_comparison,
    plot_filtered_comparison,
)

In [ ]:
# arial_font_path = "C:/brisc/fonts/arial.ttf"
arial_font_path = "/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf"

import matplotlib
import matplotlib.font_manager as fm

# set matplotlib options
if arial_font_path is not None:
    arial_prop = fm.FontProperties(fname=arial_font_path)
    plt.rcParams["font.family"] = arial_prop.get_name()
    plt.rcParams.update({"mathtext.default": "regular"})  # make math mode also Arial
    fm.fontManager.addfont(arial_font_path)

matplotlib.rcParams["pdf.fonttype"] = 42  # for pdfs

## 1. Load data

In [ ]:
processed_path = get_processed_path(
    "becalia_rabies_barseq/BRAC8498.3e/analysis/adata_q.h5ad"
)
adata = sc.read_h5ad(processed_path)
print(adata)

## 2. Run full analysis (density contour >= 50% of peak)

In [ ]:
results = run_double_labeling_analysis(adata, density_threshold=0.5)

## 3. Visualise: observed vs Poisson expected

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

plot_observed_vs_expected(
    results["observed"], results["expected"],
    results["lambda_hat"], results["chi2_test"], ax=axes[0],
    title="Linear scale",
)
plot_observed_vs_expected_log(
    results["observed"], results["expected"],
    results["lambda_hat"], ax=axes[1],
    title="Log scale",
)
plot_residuals(
    results["observed"], results["expected"], ax=axes[2],
)

fig.tight_layout()
plt.show()

In [ ]:
cm = 1 / 2.54
fontsize_dict = {"title": 7, "label": 8, "tick": 6, "legend": 6}
line_width = 0.5
line_alpha = 1

fig, ax = plt.subplots(
    figsize=(17.4 * cm, 14 * cm),
    dpi=300,
)

plot_observed_vs_expected_log(
    results["observed"], results["expected"],
    results["lambda_hat"], ax=ax,
)
plt.show()
plt.savefig("Supp_fig_barcoded_observed_vs_expected_log.pdf", bbox_inches="tight")

## 4. Show injection site density region on 2D projection

In [ ]:
from brisc.manuscript_analysis import spatial_plots_rabies as spatial

bin_image = spatial.prepare_area_labels(atlas_size=10)

In [ ]:
fontsize_dict = {"title": 8, "label": 7, "tick": 6, "legend": 6}

In [ ]:
from brisc.manuscript_analysis.double_labeling_estimation import plot_injection_site
fig, axes = plt.subplots(1,3, figsize=(17.4*cm, 7*cm))
ax = axes[0]
ylim_mm = (0, 4.8)
xlim_mm = (5.8, 11.0)

y_start, y_end = int(ylim_mm[0] * 100), int(ylim_mm[1] * 100)
x_start, x_end = int(xlim_mm[0] * 100), int(xlim_mm[1] * 100)
ax.contour(
        bin_image[y_start:y_end, x_start:x_end],
        levels=np.arange(0.5, np.max(bin_image) + 1, 0.5),
        colors="gray",
        linewidths=0.5,
        zorder=0,
        extent = [xlim_mm[0], xlim_mm[1], ylim_mm[0], ylim_mm[1]]
    )

plot_injection_site(
    adata,
    results["region_info"],
    adata_region=results["adata_region"],
    projection_axes=(2, 1),           # Coronal View
    show_density=True,                # Show contours
    density_cmap="inferno",
    show_center=False,                # Remove red cross
    barcode_inside_color="dodgerblue", # Custom colors for barcoded cells
    barcode_outside_color="k", 
    barcode_alpha=0.05, 
    layer="barcoded", 
    show_all_cells=False,
    scalebar_mm=1.0,      
    show_colorbar=True,              # <--- Enable the colorbar
    colorbar_label="Density (cells/mm³)", # Custom label if needed
    ax=ax,
    fontsize_dict=fontsize_dict
)

ax.legend(frameon=False, loc='upper left', fontsize=fontsize_dict['legend'])
if True:
    ax.set_ylim(4.8, 0)
    ax.set_xlim(11, 5.8)
    ax.set_xticks([])
    ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
ax.spines[['left', 'bottom']].set_visible(False)

plot_observed_vs_expected_log(
    results["observed"], results["expected"],
    results["lambda_hat"], ax=axes[1],
    fontsize_dict=fontsize_dict
)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

coord_filter = {0: (0.5, None), 1: (-0.5, None), 2: (0.5, None)}
projections = [(0, 2), (0, 1), (2, 1)]
titles = ["X-Z projection", "X-Y projection", "Z-Y projection"]

common = dict(
    coord_range=coord_filter,
    adata_region=results["adata_region"],
)

# Top row: density region only
for col, (proj, title) in enumerate(zip(projections, titles)):
    plot_injection_site(
        adata, results["region_info"],
        projection_axes=proj, ax=axes[0, col],
        layer="region", region_alpha=0.01,
        **common,
    )
    axes[0, col].set_title(title)

# Bottom row: barcoded cells only
for col, (proj, title) in enumerate(zip(projections, titles)):
    plot_injection_site(
        adata, results["region_info"],
        projection_axes=proj, ax=axes[1, col],
        layer="barcoded", barcode_alpha=0.04,
        **common,
    )
    axes[1, col].set_title(title)

# Invert y-axis on Z-Y panels
axes[0, 2].invert_yaxis()
axes[1, 2].invert_yaxis()

fig.tight_layout()
plt.show()

In [ ]:
from brisc.manuscript_analysis.double_labeling_estimation import plot_density_field
fig, ax = plt.subplots()
plot_density_field(
    adata, 
    projection_axes=(0, 2), # Matches the 2D projection you are viewing
    cmap="magma",           # Nice "fire" colormap for density
    ax=ax
)

## 5. Sensitivity: sweep across density thresholds

Check that results are robust to the choice of density contour level.

In [ ]:
sweep_df = sweep_density_thresholds(
    adata,
    thresholds=np.arange(0.1, 0.95, 0.1),
)
display(sweep_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_threshold_sweep(sweep_df, axes=axes)
fig.tight_layout()
plt.show()

## 6. Summary table

In [ ]:
display(results["summary_df"])

## 7. Bootstrap excess tests (k=2, k=3, k=4, combined)

In [ ]:
display(results["excess_df"])

## 8. Spot-count diagnostics

Each barcode assigned to a cell is supported by a number of decoded transcript spots.
Low spot counts could indicate spurious cross-contamination or segmentation artefacts
rather than genuine viral infection. The upstream pipeline already enforces a floor of
3 spots per barcode. Here we inspect whether the barcodes in multi-barcoded cells are
supported by fewer spots than those in single-barcoded cells.

In [ ]:
barseq_path = (
    Path("/nemo/project/proj-znamenp-barseq/processed")
    / "becalia_rabies_barseq/BRAC8498.3e/analysis"
)
cells_df = load_spots_per_barcode(
    barseq_path / "BRAC8498.3e_error_corrected_barcodes_26_cell_barcode_df.pkl"
)
spot_info = spot_count_summary(cells_df)
display(spot_info["summary_df"])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_spot_count_distributions(spot_info, ax=ax)
ax.set_title("Per-barcode spot counts: single vs multi-barcoded cells")
fig.tight_layout()
plt.show()

## 9. Sensitivity to spot-count floor

Re-run the Poisson analysis after raising the minimum number of transcript spots
required for a barcode to be counted. At floor=3 (the pipeline default), all
barcodes are kept. At floor=5 and floor=10, only barcodes with at least that
many spots contribute to `n_unique_barcodes`. If the excess of multi-barcoded
cells is driven by spurious low-spot barcodes, it should disappear at higher
floors.

In [ ]:
filtered_results = run_filtered_analysis_comparison(
    adata, cells_df,
    spot_floors=(3, 5, 10),
    density_threshold=0.5,
    n_boot=5_000,
    random_state=0,
)
display(filtered_results["comparison_df"])

In [ ]:
fig, axes = plot_filtered_comparison(filtered_results, spot_floors=(3, 5, 10))
plt.show()